# Agrupamiento (Clustering)

**SI3015 - Fundamentos de Aprendizaje Automático**

> **Datasets:** FIRE-UdeA Sintético y FIRE-UdeA Realista (Universidad de Antioquia)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


In [ ]:
random_state = 42
plt.rc('font', family='serif', size=12)


## Ejemplo Práctico

Para los datasets planteados realice el siguiente proceso de clustering:

- Halle y grafique los clusters resultantes de aplicar K-Medias con $K = 2$.
- Encuentre un buen valor de $K$ mediante el **método del codo** y grafique los nuevos clusters.
- Halle la inercia en cada caso.
- Aplique **DBSCAN** y compare con los resultados de K-Medias.

> **Nota sobre visualización:** Ambos datasets tienen más de 2 dimensiones, por lo que se usa **PCA (2 componentes)** únicamente para graficar. El clustering se aplica sobre *todas las variables numéricas escaladas*.

---
## Dataset 1 — FIRE-UdeA Sintético

Indicadores financieros sintéticos de 500 registros. Variables: `liquidez`, `dias_efectivo`, `cfo`, `participacion_ley30`, `hhi_fuentes`, `gastos_personal`, `tendencia_ingresos`.

No contiene valores nulos.

In [ ]:
# ── Carga del Dataset 1 ─────────────────────────────────────────────────
df1 = pd.read_csv('dataset_sintetico_FIRE_UdeA.csv')

# Excluir la etiqueta; en clustering NO se usa la variable objetivo
feature_cols1 = [c for c in df1.columns if c != 'label']
data1         = df1[feature_cols1].values

print(f'Shape   : {data1.shape}')
print(f'Features: {feature_cols1}')
print(f'Nulos   : {df1[feature_cols1].isnull().sum().sum()}')

# ── Proyección PCA 2D (solo para visualización) ──────────────────────────
data1_2d = PCA(n_components=2, random_state=random_state).fit_transform(
    StandardScaler().fit_transform(data1)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(data1_2d[:, 0], data1_2d[:, 1], alpha=0.6, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Dataset 1 — distribución original (PCA 2D)')
plt.tight_layout(); plt.show()


In [ ]:
# ── Pipeline de preprocesamiento DS1 ────────────────────────────────────
# Sin nulos → solo StandardScaler.
preprocessor1 = ColumnTransformer(transformers=[
    ('num', Pipeline([('scaler', StandardScaler())]), np.arange(data1.shape[1]))
])


In [ ]:
# ── K-Medias con K = 2 ──────────────────────────────────────────────────
clu_kmeans1 = Pipeline(steps=[
    ('preprocessor', preprocessor1),
    ('clustering',   KMeans(n_clusters=2, random_state=random_state, n_init='auto'))
])
clu_kmeans1.fit(data1)

print(f'Inercia (K=2): {clu_kmeans1["clustering"].inertia_:.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(data1_2d[:, 0], data1_2d[:, 1],
               c=clu_kmeans1['clustering'].labels_,
               cmap='tab10', alpha=0.8, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Dataset 1 — K-Medias K=2 (PCA 2D)')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()


In [ ]:
# ── Método del Codo ─────────────────────────────────────────────────────
inert1  = []
k_range = list(range(1, 11))

for k in k_range:
    tmp = Pipeline(steps=[
        ('preprocessor', preprocessor1),
        ('clustering',   KMeans(n_clusters=k, random_state=random_state, n_init='auto'))
    ])
    tmp.fit(data1)
    inert1.append(tmp['clustering'].inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, inert1, 'o-', color='steelblue')
ax.set_xlabel('K'); ax.set_ylabel('Inercia')
ax.set_title('Dataset 1 — Método del Codo')
plt.tight_layout(); plt.show()


Identificar el "codo" (punto de mayor inflexión) para elegir el $K$ óptimo.

In [ ]:
# ── K-Medias con K óptimo ───────────────────────────────────────────────
K_optimo1 = 2  # <-- ajustar según el codo observado

clu_kmeans1_opt = Pipeline(steps=[
    ('preprocessor', preprocessor1),
    ('clustering',   KMeans(n_clusters=K_optimo1, random_state=random_state, n_init='auto'))
])
clu_kmeans1_opt.fit(data1)

print(f'Inercia (K={K_optimo1}): {clu_kmeans1_opt["clustering"].inertia_:.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(data1_2d[:, 0], data1_2d[:, 1],
               c=clu_kmeans1_opt['clustering'].labels_,
               cmap='tab10', alpha=0.8, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'Dataset 1 — K-Medias K={K_optimo1} óptimo (PCA 2D)')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()


### DBSCAN — Dataset 1

> **Nota pedagógica:** DBSCAN busca regiones de alta densidad separadas por zonas de baja densidad. Los datos financieros tabulares en múltiples dimensiones tienden a distribuirse de forma relativamente uniforme (sin "islas" densas bien separadas), por lo que DBSCAN suele encontrar un único cluster grande o mucho ruido. Esto es distinto a datos 2D con formas como `make_circles` o `make_moons`, donde DBSCAN destaca. Experimentar con `eps` y `min_samples` permite observar este comportamiento.

In [ ]:
# ── DBSCAN — Dataset 1 ──────────────────────────────────────────────────
# eps=2.0, min_samples=5: parámetros calibrados sobre el espacio normalizado.
clu_dbscan1 = Pipeline(steps=[
    ('preprocessor', preprocessor1),
    ('clustering',   DBSCAN(eps=2.0, min_samples=5))
])
clu_dbscan1.fit(data1)
labels_db1 = clu_dbscan1['clustering'].labels_

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(data1_2d[:, 0], data1_2d[:, 1], c=labels_db1,
               cmap='tab10', alpha=0.8, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Dataset 1 — DBSCAN (PCA 2D)  |  −1 = ruido')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()

unique, counts = np.unique(labels_db1, return_counts=True)
for u, c in zip(unique, counts):
    etiqueta = 'Ruido' if u == -1 else f'Cluster {u}'
    print(f'  {etiqueta}: {c} puntos')


---
## Dataset 2 — FIRE-UdeA Realista

Datos financieros de **8 unidades académicas/administrativas** de la Universidad de Antioquia, periodo **2016–2025** (80 registros, 13 variables financieras).

| Columna | Rol |
|---------|-----|
| `anio`, `unidad` | Identificadores — se **excluyen** |
| `label` | Etiqueta de clasificación — se **excluye** en clustering |
| demás columnas | Features numéricas para clustering |

> Contiene **valores nulos** en varias variables financieras. Se **imputan con la mediana** (robusta a outliers), tal como se determinó en el EDA previo.

In [ ]:
# ── Carga del Dataset 2 ─────────────────────────────────────────────────
df2 = pd.read_csv('dataset_sintetico_FIRE_UdeA_realista.csv')

drop_cols2    = ['anio', 'unidad', 'label']
feature_cols2 = [c for c in df2.columns if c not in drop_cols2]
data2         = df2[feature_cols2].values

print(f'Shape   : {data2.shape}')
print(f'Features ({len(feature_cols2)}): {feature_cols2}')

print('\nValores nulos por columna:')
nulos = df2[feature_cols2].isnull().sum()
print(nulos[nulos > 0].to_string())

# ── Proyección PCA 2D para visualización ─────────────────────────────────
data2_imp = SimpleImputer(strategy='median').fit_transform(data2)
data2_2d  = PCA(n_components=2, random_state=random_state).fit_transform(
    StandardScaler().fit_transform(data2_imp)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(data2_2d[:, 0], data2_2d[:, 1], alpha=0.7, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Dataset 2 — distribución original (PCA 2D)')
plt.tight_layout(); plt.show()


In [ ]:
# ── Pipeline de preprocesamiento DS2 ────────────────────────────────────
# Con nulos → imputar mediana ANTES de escalar.
preprocessor2 = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), np.arange(data2.shape[1]))
])


In [ ]:
# ── K-Medias con K = 2 ──────────────────────────────────────────────────
clu_kmeans2 = Pipeline(steps=[
    ('preprocessor', preprocessor2),
    ('clustering',   KMeans(n_clusters=2, random_state=random_state, n_init='auto'))
])
clu_kmeans2.fit(data2)

print(f'Inercia (K=2): {clu_kmeans2["clustering"].inertia_:.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(data2_2d[:, 0], data2_2d[:, 1],
               c=clu_kmeans2['clustering'].labels_,
               cmap='tab10', alpha=0.8, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Dataset 2 — K-Medias K=2 (PCA 2D)')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()


In [ ]:
# ── Método del Codo ─────────────────────────────────────────────────────
inert2  = []
k_range = list(range(1, 11))

for k in k_range:
    tmp = Pipeline(steps=[
        ('preprocessor', preprocessor2),
        ('clustering',   KMeans(n_clusters=k, random_state=random_state, n_init='auto'))
    ])
    tmp.fit(data2)
    inert2.append(tmp['clustering'].inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, inert2, 'o-', color='darkorange')
ax.set_xlabel('K'); ax.set_ylabel('Inercia')
ax.set_title('Dataset 2 — Método del Codo')
plt.tight_layout(); plt.show()


Identificar el "codo" para elegir el $K$ óptimo.

In [ ]:
# ── K-Medias con K óptimo ───────────────────────────────────────────────
K_optimo2 = 3  # <-- ajustar según el codo observado

clu_kmeans2_opt = Pipeline(steps=[
    ('preprocessor', preprocessor2),
    ('clustering',   KMeans(n_clusters=K_optimo2, random_state=random_state, n_init='auto'))
])
clu_kmeans2_opt.fit(data2)

print(f'Inercia (K={K_optimo2}): {clu_kmeans2_opt["clustering"].inertia_:.2f}')

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(data2_2d[:, 0], data2_2d[:, 1],
               c=clu_kmeans2_opt['clustering'].labels_,
               cmap='tab10', alpha=0.8, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'Dataset 2 — K-Medias K={K_optimo2} óptimo (PCA 2D)')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()


### DBSCAN — Dataset 2

> En 13 dimensiones, la distancia entre puntos crece (maldición de la dimensionalidad), lo que dificulta encontrar regiones densas bien definidas. Con `eps=4.0` (espacio normalizado) y `min_samples=4` se obtienen resultados representativos del comportamiento de DBSCAN en datos financieros.

In [ ]:
# ── DBSCAN — Dataset 2 ──────────────────────────────────────────────────
clu_dbscan2 = Pipeline(steps=[
    ('preprocessor', preprocessor2),
    ('clustering',   DBSCAN(eps=4.0, min_samples=4))
])
clu_dbscan2.fit(data2)
labels_db2 = clu_dbscan2['clustering'].labels_

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(data2_2d[:, 0], data2_2d[:, 1], c=labels_db2,
               cmap='tab10', alpha=0.8, edgecolors='k', linewidths=0.3)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Dataset 2 — DBSCAN (PCA 2D)  |  −1 = ruido')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()

unique, counts = np.unique(labels_db2, return_counts=True)
for u, c in zip(unique, counts):
    etiqueta = 'Ruido' if u == -1 else f'Cluster {u}'
    print(f'  {etiqueta}: {c} puntos')
